In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
df = pd.read_csv("military_raw_data.csv")

print(df.shape)
df.head()

(145, 54)


,Country,total_military_manpower,fit_for_service,population_reaching_military_age_annually,active_personnel,reserve_personnel,paramilitary,total_military_aircraft,fighter_aircraft,attack_aircraft,...,natural_gas_production_cum,natural_gas_consumption_cum,proven_natural_gas_reserves_cum,coal_production_cum,coal_consumption_mt,proven_coal_reserves_cum,total_land_area_sq_km,coastline_coverage_km,border_coverage_km,waterway_coverage_km
0,China,"764,123,366","626,864,169","19,810,606","2,035,000","510,000","625,000","3,529","1,443",371,...,"225,341,000,000 Cu.M","366,160,000,000 Cu.M","6,654,000,000,000 Cu.M","4,805,000,000 Cu.M","5,191,000,000 mt","157,041,000,000 Cu.M","9,596,960 km","14,500 km","22,457 km","27,700 km"
1,India,"662,290,299","522,786,598","23,955,181","1,431,000","1,000,000","2,527,000","2,183",476,124,...,"33,170,000,000 Cu.M","58,867,000,000 Cu.M","1,381,000,000,000 Cu.M","1,020,000,000 Cu.M","1,262,000,000 mt","127,727,000,000 Cu.M","3,287,263 km","7,000 km","13,888 km","14,500 km"
2,United States,"150,463,900","124,816,644","4,445,524","1,333,030","799,500",0,"13,032","1,791",926,...,"1,029,000,000,000 Cu.M","914,301,000,000 Cu.M","13,402,000,000,000 Cu.M","534,234,000 Cu.M","495,156,000 mt","247,883,000,000 Cu.M","9,833,517 km","19,924 km","12,002 km","41,009 km"
3,Indonesia,"137,965,608","114,595,923","4,786,562","404,500","401,000","250,000",460,41,34,...,"57,410,000,000 Cu.M","36,061,000,000 Cu.M","1,408,000,000,000 Cu.M","783,453,000 Cu.M","281,159,000 mt","35,055,000,000 Cu.M","1,904,569 km","54,716 km","2,958 km","21,579 km"
4,Nigeria,"125,475,979","90,437,404","4,261,448","230,000",0,"50,000",159,14,24,...,"39,951,000,000 Cu.M","20,719,000,000 Cu.M","5,761,000,000,000 Cu.M","1,322,000 Cu.M","1,326,000 mt","2,144,000,000 Cu.M","923,768 km",853 km,"4,477 km","8,600 km"


In [3]:
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_", regex=False)
      .str.replace("-", "_", regex=False)
)

print(df.columns.tolist())

['country', 'total_military_manpower', 'fit_for_service', 'population_reaching_military_age_annually', 'active_personnel', 'reserve_personnel', 'paramilitary', 'total_military_aircraft', 'fighter_aircraft', 'attack_aircraft', 'transport_aircraft', 'trainer_aircraft', 'special_mission_aircraft', 'tanker_aircraft', 'total_military_helicopters', 'attack_helicopters', 'tanks', 'armored_fighting_vehicles', 'self_propelled_artillery', 'towed_artillery', 'rocket_projectors', 'total_naval_fleet', 'total_naval_fleet_tonnage_mt', 'aircraft_carriers', 'helicopter_carriers', 'submarines', 'destroyers', 'frigates', 'corvettes', 'coastal_patrol_craft', 'mine_warfare_craft', 'defense_budget_usd', 'external_debt_usd', 'purchasing_power_parity_usd', 'foreign_exchange_and_gold_reserves_usd', 'total_serviceable_airports', 'labour_force', 'major_ports_and_terminals', 'total_merchant_marine_fleet', 'railway_coverage_km', 'roadway_coverage_km', 'oil_production_bbl', 'oil_consumption_bbl', 'proven_oil_reserv

In [4]:
def clean_value(x):

    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    # Missing values
    if x in ["", "-", "--", "N/A", "NA", "None", "null"]:
        return np.nan

    # Remove commas
    x = x.replace(",", "")

    # Remove symbols
    x = x.replace("$", "")
    x = x.replace("%", "")
    x = x.replace("+", "")

    # Remove ALL alphabetic characters
    # Examples:
    # Cu.M
    # USD
    # Km
    # Km²
    # MT
    # BBL

    x = re.sub(r"[A-Za-z²./]+", "", x)

    x = x.strip()

    if x == "":
        return np.nan

    return x

In [5]:
for col in df.columns:

    if col.lower() != "country":

        df[col] = df[col].apply(clean_value)

        df[col] = pd.to_numeric(df[col], errors="coerce")

In [6]:
print(df.isnull().sum())

country                                       0
total_military_manpower                       0
fit_for_service                               0
population_reaching_military_age_annually     0
active_personnel                              0
reserve_personnel                             0
paramilitary                                  0
total_military_aircraft                       0
fighter_aircraft                              0
attack_aircraft                               0
transport_aircraft                            0
trainer_aircraft                              0
special_mission_aircraft                      0
tanker_aircraft                               0
total_military_helicopters                    0
attack_helicopters                            0
tanks                                         0
armored_fighting_vehicles                     0
self_propelled_artillery                      0
towed_artillery                               0
rocket_projectors                       

In [7]:
numeric_cols = df.select_dtypes(include="number").columns

for col in numeric_cols:

    if df[col].isna().sum() > 0:

        median = df[col].median()

        df[col] = df[col].fillna(median)

In [8]:
country_column = [col for col in df.columns if col.lower() == "country"][0]

df.drop_duplicates(subset=country_column, inplace=True)

In [9]:
for col in numeric_cols:

    df[col] = df[col].round().astype("Int64")

In [10]:
df[[
    "country",
    "natural_gas_production_cum",
    "natural_gas_consumption_cum",
    "proven_natural_gas_reserves_cum"
]].head()

,country,natural_gas_production_cum,natural_gas_consumption_cum,proven_natural_gas_reserves_cum
0,China,225341000000,366160000000,6654000000000
1,India,33170000000,58867000000,1381000000000
2,United States,1029000000000,914301000000,13402000000000
3,Indonesia,57410000000,36061000000,1408000000000
4,Nigeria,39951000000,20719000000,5761000000000


In [13]:
df.to_csv("military_cleaned.csv", index=False)

print("military_cleaned.csv created successfully.")

military_cleaned.csv created successfully.
